# LLM Corpus Annotation — Claude Haiku

Annotates every sentence in `data/processed/corpus.csv` with one or more framing labels
using `claude-haiku-4-5` and the project annotation guidelines.

**Output:** `data/annotation/labeled_full.csv`  
**System prompt:** loaded live from `docs/annotation_guidelines.md` + the exact template
specified in those guidelines.  
**Prompt caching:** the system prompt is cached with `cache_control: ephemeral`,
keeping costs low across the full run.  
**Checkpointing:** progress is saved every `CHECKPOINT_EVERY` sentences — safe to
interrupt and resume; already-annotated sentences are skipped automatically.

In [ ]:
import json
import logging
import random
import sys
import time
from pathlib import Path

import anthropic
import nltk
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from src.config import (
    ANTHROPIC_API_KEY,
    CORPUS_CSV,
    DATA_ANNOTATION,
    DOCS,
    FRAMES,
    LABELED_CSV,
)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 256        # labels JSON is always short
CHECKPOINT_EVERY = 50   # save to disk after every N new annotations
MAX_RETRIES = 6
BASE_BACKOFF = 2.0      # seconds; doubles on each rate-limit retry

# Maps annotation frame names to binary output column names
FRAME_COLS: dict[str, str] = {
    "Innovation/Progress":   "innovation_progress",
    "Economic Benefit":      "economic_benefit",
    "Risk/Harm":             "risk_harm",
    "Regulation/Governance": "regulation_governance",
    "Existential/AGI":       "existential_agi",
    "None":                  "none",
}

META_COLS = [
    "doc_id", "actor", "actor_type", "positioning",
    "context", "platform", "date", "post_chatgpt",
]

In [ ]:
# ── Build system prompt ────────────────────────────────────────────────────────
# Uses the exact LLM Labeling System Prompt Template from annotation_guidelines.md,
# with the full guidelines text inserted in place of the [INSERT ...] placeholder.
# Reading at runtime keeps the prompt in sync if the guidelines are updated.

GUIDELINES_PATH = DOCS / "annotation_guidelines.md"
guidelines_text = GUIDELINES_PATH.read_text(encoding="utf-8")

_TEMPLATE = (
    "You are a research assistant for a computational linguistics project studying how AI\n"
    "industry actors frame artificial intelligence across different contexts.\n"
    "\n"
    "Label each sentence with ONE OR MORE of these frames:\n"
    "  1. Innovation/Progress\n"
    "  2. Economic Benefit\n"
    "  3. Risk/Harm\n"
    "  4. Regulation/Governance\n"
    "  5. Existential/AGI\n"
    "\n"
    "Labeling rules:\n"
    "- A sentence can have multiple labels if more than one frame applies\n"
    "- Use None if no frame applies (procedural text, vague claims, attributions)\n"
    "- The harm in Risk/Harm must be concrete and near-term, not vague concern\n"
    "- Regulation/Governance requires an explicit institutional mechanism or law\n"
    "- Existential/AGI requires civilizational scale or explicit AGI/superintelligence reference\n"
    "- Do NOT label sentences as Risk/Harm just because they mention AI risks vaguely\n"
    "- Do NOT label sentences as Regulation/Governance just because they mention responsibility\n"
    "\n"
    'Return ONLY valid JSON. No preamble, no explanation, no markdown.\n'
    'Format: {{"labels": ["Innovation/Progress", "Economic Benefit"]}}\n'
    'If None: {{"labels": ["None"]}}\n'
    "\n"
    "{guidelines}"
)

SYSTEM_PROMPT = _TEMPLATE.format(guidelines=guidelines_text)
print(f"System prompt: {len(SYSTEM_PROMPT):,} chars  "
      f"(caching kicks in at ~4,000 chars — we are well above threshold)")

In [ ]:
# ── Load corpus and tokenize into sentences ───────────────────────────────────
corpus = pd.read_csv(CORPUS_CSV, dtype={"doc_id": str})
print(f"Corpus: {len(corpus):,} documents, {corpus['actor'].nunique()} actors")

rows = []
for _, doc in corpus.iterrows():
    sentences = nltk.sent_tokenize(str(doc["text"]))
    for idx, sent in enumerate(sentences):
        sent = sent.strip()
        if not sent:
            continue
        record = {col: doc[col] for col in META_COLS}
        record["sentence_idx"] = idx
        record["sentence"] = sent
        rows.append(record)

sentences_df = pd.DataFrame(rows)
print(f"Sentences: {len(sentences_df):,} total  "
      f"(avg {len(sentences_df) / len(corpus):.1f} per document)")

In [ ]:
# ── Load checkpoint ────────────────────────────────────────────────────────────
DATA_ANNOTATION.mkdir(parents=True, exist_ok=True)

if LABELED_CSV.exists():
    done = pd.read_csv(LABELED_CSV, dtype={"doc_id": str})
    done_keys = set(
        done["doc_id"].astype(str) + "_" + done["sentence_idx"].astype(str)
    )
    log.info("Resuming from checkpoint: %d sentences already annotated", len(done))
else:
    done = pd.DataFrame()
    done_keys = set()
    log.info("Starting fresh annotation run")

sentences_df["_key"] = (
    sentences_df["doc_id"].astype(str) + "_"
    + sentences_df["sentence_idx"].astype(str)
)
to_annotate = (
    sentences_df[~sentences_df["_key"].isin(done_keys)]
    .drop(columns="_key")
    .reset_index(drop=True)
)
sentences_df.drop(columns="_key", inplace=True)

print(f"Remaining: {len(to_annotate):,} sentences to annotate")

In [ ]:
# ── Annotation helpers ─────────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

VALID_LABELS = set(FRAMES + ["None"])


def parse_labels(text: str) -> list[str]:
    """Parse the JSON label list from a model response."""
    text = text.strip()
    # strip markdown code fences if the model adds them despite instructions
    if text.startswith("```"):
        parts = text.split("```")
        text = parts[1].lstrip("json").strip() if len(parts) > 1 else text
    try:
        data = json.loads(text)
        labels = [l for l in data.get("labels", []) if l in VALID_LABELS]
        return labels if labels else ["None"]
    except (json.JSONDecodeError, AttributeError, TypeError):
        log.warning("JSON parse failure: %r", text[:120])
        return ["None"]


def annotate_sentence(sentence: str) -> list[str]:
    """Call Claude Haiku with exponential backoff; return label list."""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=[
                    {
                        "type": "text",
                        "text": SYSTEM_PROMPT,
                        "cache_control": {"type": "ephemeral"},
                    }
                ],
                messages=[{"role": "user", "content": sentence}],
            )
            return parse_labels(response.content[0].text)
        except anthropic.RateLimitError:
            wait = BASE_BACKOFF * (2 ** attempt) + random.uniform(0, 1)
            log.warning(
                "Rate limit — sleeping %.1fs (attempt %d/%d)",
                wait, attempt + 1, MAX_RETRIES,
            )
            time.sleep(wait)
        except anthropic.APIStatusError as exc:
            if exc.status_code >= 500:
                wait = BASE_BACKOFF * (2 ** attempt)
                log.warning("Server error %d — sleeping %.1fs", exc.status_code, wait)
                time.sleep(wait)
            else:
                raise
    log.error("Max retries exceeded for: %r", sentence[:80])
    return ["None"]

In [ ]:
# ── Main annotation loop ───────────────────────────────────────────────────────
results: list[dict] = done.to_dict("records") if not done.empty else []
n_new = 0
total = len(to_annotate)

for _, row in to_annotate.iterrows():
    labels = annotate_sentence(row["sentence"])

    record: dict = {col: row[col] for col in META_COLS + ["sentence_idx", "sentence"]}
    record["labels"] = ",".join(labels)
    for frame, col in FRAME_COLS.items():
        record[col] = int(frame in labels)
    results.append(record)
    n_new += 1

    if n_new % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(LABELED_CSV, index=False)
        log.info(
            "Checkpoint: %d/%d new | %d/%d total",
            n_new, total, len(results), len(sentences_df),
        )

# Final save
pd.DataFrame(results).to_csv(LABELED_CSV, index=False)
log.info(
    "Done. %d sentences annotated and saved to %s",
    len(results), LABELED_CSV,
)

In [ ]:
# ── Summary statistics ─────────────────────────────────────────────────────────
labeled = pd.read_csv(LABELED_CSV, dtype={"doc_id": str})
bin_cols = list(FRAME_COLS.values())

print(f"Total sentences annotated : {len(labeled):,}")
print(f"Documents covered         : {labeled['doc_id'].nunique():,}")
print()

counts = labeled[bin_cols].sum().sort_values(ascending=False)
pct = (counts / len(labeled) * 100).round(1)
summary = pd.DataFrame({"sentences": counts.astype(int), "% of total": pct})
print("Label distribution:")
print(summary.to_string())
print()

none_rate = labeled["none"].mean() * 100
multi_rate = (labeled[bin_cols].sum(axis=1) > 1).mean() * 100
print(f"None rate        : {none_rate:.1f}%  (guideline target: 20–30%)")
print(f"Multi-label rate : {multi_rate:.1f}%")
print()

# Per-actor breakdown
print("Label rates by actor:")
actor_summary = (
    labeled.groupby("actor")[bin_cols]
    .mean()
    .mul(100)
    .round(1)
    .sort_values("innovation_progress", ascending=False)
)
print(actor_summary.to_string())